In [1]:
# -*- coding: utf-8 -*-
# =====================================================================================
# Last.fm Trusted Node Election
# Scenario-1: Global Nearest by Friend Distance -> Leader by MIN SUM DISTANCE
# NO FALLBACK / NO PENALTY / NO NORMALIZATION
#
# Inputs:
#   1) Last.fm discovered community files:
#      comm_U_..._I_..._<community_id>.txt
#
#   2) Last.fm friend-friend Jaccard similarity file:
#      lastfm_friends_jaccard_sim_nonzero.csv
#
# Required similarity columns:
#   friend_1, friend_2, jaccard_sim
#
# Main rule:
#   d(u,v) = 1 - jaccard_sim(u,v)
#
# Winner selection:
#   1) MIN sum_dist
#   2) tie-break: coverage_count DESC
#   3) tie-break: candidate_id ASC
# =====================================================================================

import os
import re
import numpy as np
import pandas as pd
from datetime import datetime
from collections import defaultdict

# =====================================================================================
# Cell 0 — Paths + Configuration
# =====================================================================================

# Results will be stored here
OP_ROOT = r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\2_التصويت"

RUN_ID = datetime.now().strftime("RUN_%Y%m%d_%H%M%S")

project_dir = os.path.join(OP_ROOT, RUN_ID)
outputs_dir = os.path.join(project_dir, "outputs")
os.makedirs(outputs_dir, exist_ok=True)

# Friend-friend Jaccard similarity file produced by the previous code
friends_sim_csv = r"D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\lastfm_friend_sim_mtrx\lastfm_friends_jaccard_sim_nonzero.csv"

# Last.fm discovered communities
COMMUNITY_FILES = [
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_136_I_1884_60.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_81_I_1572_99.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_66_I_1051_30.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_58_I_1010_40.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_57_I_1188_38.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_55_I_717_211.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_44_I_1227_107.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_38_I_942_28.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_28_I_817_114.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_25_I_787_104.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_30_I_607_204.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_29_I_639_506.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_26_I_675_44.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_21_I_560_348.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_18_I_559_265.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_16_I_554_5.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_16_I_513_163.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_15_I_535_167.txt",
    r"D:\PHD\datasetsV2\hetrec2011-lastfm-2k\artis_info_json معتمد\Pred_users\COMM_TXT_alpha_0.340\comm_U_14_I_505_150.txt",
]

interest_files = COMMUNITY_FILES

CHUNKSIZE = 400_000
EPS = 1e-12

WRITE_DISTANCE_EDGELIST = True
WRITE_WITHIN_I_MATRICES = True
WRITE_NOMINATION_FILES = True
WRITE_CANDIDATE_FILES = True

print("[INFO] RUN_ID:", RUN_ID)
print("[INFO] project_dir:", project_dir)
print("[INFO] outputs_dir:", outputs_dir)
print("[INFO] friends_sim_csv:", friends_sim_csv)
print("[INFO] number of community files:", len(interest_files))


# =====================================================================================
# Cell 1 — Helper Functions + Path Checks
# =====================================================================================

def safe_strip_cols(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = df.columns.astype(str).str.strip()
    return df


def ensure_path_exists(p: str, name: str = "") -> str:
    if os.path.exists(p):
        return p
    raise FileNotFoundError(f"[ERROR] Missing {name}: {p}")


def parse_interest_comm_id(path: str) -> int:
    """
    Last.fm format:
        comm_U_136_I_1884_60.txt
    The last number is used as the community ID.

    Also supports Douban-like format:
        003_comm_8_U104_I3510.txt
    """
    base = os.path.basename(path)

    # Last.fm format
    m = re.search(r"comm_U_\d+_I_\d+_(\d+)", base)
    if m:
        return int(m.group(1))

    # Douban-like format
    m = re.search(r"comm_(\d+)", base)
    if m:
        return int(m.group(1))

    raise ValueError(f"Cannot parse community ID from filename: {path}")


def load_interest_file_users(fp: str):
    """
    Reads a Last.fm community file and extracts users.

    The voting process needs only User IDs.
    Supported user column names:
        User, user, userID, UserID, Node, node

    If no header is detected, the first column is treated as the user column.
    """
    try:
        df = pd.read_csv(fp, sep=r"\s+", engine="python")
        df = safe_strip_cols(df)
    except Exception:
        df = pd.read_csv(fp, sep=r"\s+", header=None, engine="python")

    possible_user_cols = [
        "User", "user", "userID", "UserID",
        "Node", "node"
    ]

    user_col = None
    if isinstance(df.columns[0], str):
        for col in possible_user_cols:
            if col in df.columns:
                user_col = col
                break

    # Fallback: first column if no known user column exists
    if user_col is None:
        df_no_header = pd.read_csv(
            fp,
            sep=r"\s+",
            header=None,
            engine="python"
        )

        users = sorted(
            df_no_header.iloc[:, 0]
            .dropna()
            .astype(int)
            .unique()
            .tolist()
        )

        return users, {
            "rows": int(len(df_no_header)),
            "n_users": int(len(users)),
            "n_items": None,
            "user_col": "column_0_no_header"
        }

    users = sorted(
        df[user_col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    possible_item_cols = [
        "Movie", "movie",
        "Artist", "artist",
        "artistID", "ArtistID",
        "Item", "item",
        "itemID", "ItemID"
    ]

    item_col = None
    for col in possible_item_cols:
        if col in df.columns:
            item_col = col
            break

    if item_col is not None:
        n_items = int(df[item_col].nunique())
    else:
        n_items = None

    return users, {
        "rows": int(len(df)),
        "n_users": int(len(users)),
        "n_items": n_items,
        "user_col": user_col
    }


friends_sim_csv = ensure_path_exists(friends_sim_csv, "friends_sim_csv")

for fp in interest_files:
    ensure_path_exists(fp, f"interest_file: {os.path.basename(fp)}")

# Output folders
step1_dir = os.path.join(outputs_dir, "step1_interest")
vote_dir = os.path.join(outputs_dir, "scenario1_global_friend_distance_sumdist_no_fallback")
nom_dir = os.path.join(vote_dir, "nominations_per_interest")
cand_dir = os.path.join(vote_dir, "candidate_scores_per_interest")
final_dir = os.path.join(vote_dir, "final")
mat_dir = os.path.join(vote_dir, "friend_distance_matrices_within_I")

for d in [step1_dir, vote_dir, nom_dir, cand_dir, final_dir]:
    os.makedirs(d, exist_ok=True)

if WRITE_WITHIN_I_MATRICES:
    os.makedirs(mat_dir, exist_ok=True)

print("[OK] Input paths verified. Output folders created.")


# =====================================================================================
# Cell 2 — Extract Interest Community Membership
# =====================================================================================

interest_rows = []
files_meta = []

for fp in interest_files:
    ic = parse_interest_comm_id(fp)

    users, meta = load_interest_file_users(fp)

    for u in users:
        interest_rows.append({
            "interest_comm_id": int(ic),
            "User": int(u)
        })

    files_meta.append({
        "interest_comm_id": int(ic),
        "file_path": fp,
        "rows": int(meta["rows"]),
        "n_users": int(meta["n_users"]),
        "n_items": meta["n_items"],
        "user_col": meta["user_col"]
    })

interest_membership = (
    pd.DataFrame(interest_rows)
    .sort_values(["interest_comm_id", "User"])
    .reset_index(drop=True)
)

interest_sizes = (
    interest_membership
    .groupby("interest_comm_id")["User"]
    .nunique()
    .reset_index(name="interest_comm_size")
    .sort_values("interest_comm_id")
    .reset_index(drop=True)
)

df_files_meta = (
    pd.DataFrame(files_meta)
    .sort_values("interest_comm_id")
    .reset_index(drop=True)
)

interest_membership.to_csv(
    os.path.join(step1_dir, "lastfm_interest_membership.csv"),
    index=False,
    encoding="utf-8-sig"
)

interest_sizes.to_csv(
    os.path.join(step1_dir, "lastfm_interest_community_sizes.csv"),
    index=False,
    encoding="utf-8-sig"
)

df_files_meta.to_csv(
    os.path.join(step1_dir, "lastfm_interest_files_meta.csv"),
    index=False,
    encoding="utf-8-sig"
)

interest_groups = (
    interest_membership
    .groupby("interest_comm_id")["User"]
    .apply(lambda x: sorted(x.astype(int).tolist()))
    .to_dict()
)

all_users = set()
for ic, users in interest_groups.items():
    all_users.update(users)

print("[OK] Interest communities:", len(interest_groups))
print("[OK] Total unique interest users:", len(all_users))
print("\n[INTEREST COMMUNITY SIZES]")
print(interest_sizes.to_string(index=False))


# =====================================================================================
# Cell 3 — Scan Friend Similarity Edge List
# =====================================================================================

hdr = safe_strip_cols(pd.read_csv(friends_sim_csv, nrows=0))

required_cols = {"friend_1", "friend_2", "jaccard_sim"}

if not required_cols.issubset(set(hdr.columns)):
    raise ValueError(
        f"[ERROR] friends_sim_csv must contain {required_cols}, "
        f"but found {set(hdr.columns)}"
    )

usecols = ["friend_1", "friend_2", "jaccard_sim"]

# Best global nearest neighbor for each interest user
best_nb = {int(u): None for u in all_users}
best_sim = {int(u): -1.0 for u in all_users}
best_dist = {int(u): None for u in all_users}

present_users = set()

# adj_dist[u][v] = distance between interest user u and node v
adj_dist = defaultdict(dict)

dist_path = os.path.join(vote_dir, "lastfm_friends_distance_edgelist_used_by_voting.csv")
dist_writer = None

if WRITE_DISTANCE_EDGELIST:
    dist_writer = open(dist_path, "w", encoding="utf-8")
    dist_writer.write("friend_1,friend_2,jaccard_sim,jaccard_dist\n")

for chunk in pd.read_csv(
    friends_sim_csv,
    chunksize=CHUNKSIZE,
    usecols=usecols
):
    chunk = safe_strip_cols(chunk)

    chunk["friend_1"] = chunk["friend_1"].astype(np.int64)
    chunk["friend_2"] = chunk["friend_2"].astype(np.int64)
    chunk["jaccard_sim"] = chunk["jaccard_sim"].astype(float)

    if WRITE_DISTANCE_EDGELIST:
        tmp = chunk.copy()
        tmp["jaccard_dist"] = 1.0 - tmp["jaccard_sim"]
        tmp.to_csv(dist_writer, index=False, header=False)

    a = chunk["friend_1"].to_numpy()
    b = chunk["friend_2"].to_numpy()
    w = chunk["jaccard_sim"].to_numpy()

    for u, v, ww in zip(a, b, w):
        u = int(u)
        v = int(v)
        ww = float(ww)

        if u == v:
            continue

        d = 1.0 - ww

        # Case 1: u is an interest user
        if u in best_nb:
            present_users.add(u)

            prev = adj_dist[u].get(v, None)
            if (prev is None) or (d < prev):
                adj_dist[u][v] = d

            curw = best_sim[u]
            curnb = best_nb[u]

            if (ww > curw) or (
                abs(ww - curw) <= EPS and
                (curnb is None or v < curnb)
            ):
                best_sim[u] = ww
                best_nb[u] = v
                best_dist[u] = d

        # Case 2: v is an interest user
        if v in best_nb:
            present_users.add(v)

            prev = adj_dist[v].get(u, None)
            if (prev is None) or (d < prev):
                adj_dist[v][u] = d

            curw = best_sim[v]
            curnb = best_nb[v]

            if (ww > curw) or (
                abs(ww - curw) <= EPS and
                (curnb is None or u < curnb)
            ):
                best_sim[v] = ww
                best_nb[v] = u
                best_dist[v] = d

if dist_writer:
    dist_writer.close()

print("[OK] Friend similarity scan complete.")
print("[INFO] Interest users found in friend similarity file:", len(present_users))
print("[INFO] Interest users not found, therefore abstain:", len(all_users) - len(present_users))


# =====================================================================================
# Cell 4 — Nomination + Candidate Scoring + Trusted Node Election
# =====================================================================================

summary_rows = []

for ic in sorted(interest_groups.keys()):
    I = [int(u) for u in interest_groups[ic]]
    Iset = set(I)
    total = len(I)

    # -----------------------------------------------------------------
    # Step A: Nominations
    # -----------------------------------------------------------------
    nomination_rows = []
    abstain_count = 0

    for u in I:
        v = best_nb[u]

        if v is None:
            abstain_count += 1

            nomination_rows.append({
                "interest_comm_id": int(ic),
                "voter": int(u),
                "chosen_candidate": None,
                "chosen_sim": None,
                "chosen_dist": None,
                "abstained": True
            })

        else:
            nomination_rows.append({
                "interest_comm_id": int(ic),
                "voter": int(u),
                "chosen_candidate": int(v),
                "chosen_sim": float(best_sim[u]),
                "chosen_dist": float(best_dist[u]),
                "abstained": False
            })

    df_nom = pd.DataFrame(nomination_rows)

    if WRITE_NOMINATION_FILES:
        df_nom.to_csv(
            os.path.join(nom_dir, f"lastfm_nominations_global_friend_comm_{int(ic)}.csv"),
            index=False,
            encoding="utf-8-sig"
        )

        df_nom.to_excel(
            os.path.join(nom_dir, f"lastfm_nominations_global_friend_comm_{int(ic)}.xlsx"),
            index=False
        )

    # -----------------------------------------------------------------
    # Step B: Candidate Set
    # -----------------------------------------------------------------
    df_nom_valid = df_nom[df_nom["abstained"] == False].copy()

    if len(df_nom_valid) == 0:
        summary_rows.append({
            "interest_comm_id": int(ic),
            "interest_size": int(total),
            "nominating_voters": 0,
            "abstain_count": int(abstain_count),
            "nomination_coverage": 0.0,
            "unique_nominees": 0,
            "leader": None,
            "leader_sum_dist": None,
            "leader_coverage_count": 0,
            "leader_coverage_ratio": 0.0,
            "leader_nominated_by_count": 0,
            "leader_in_IC": None,
            "decision_basis": "no_nominee",
            "tie_group_size": 0,
            "runner_up": None,
            "runner_up_sum_dist": None,
            "runner_up_coverage_count": None,
            "sum_margin": None,
            "coverage_margin": None
        })

        continue

    candidates = sorted(
        df_nom_valid["chosen_candidate"]
        .astype(int)
        .unique()
        .tolist()
    )

    nominated_by_count_map = (
        df_nom_valid
        .groupby("chosen_candidate")["voter"]
        .count()
        .astype(int)
        .to_dict()
    )

    # -----------------------------------------------------------------
    # Step C: Candidate Scoring
    # -----------------------------------------------------------------
    cand_rows = []

    for c in candidates:
        c = int(c)

        sum_dist = 0.0
        coverage_count = 0
        missing_count = 0

        for u in I:
            u = int(u)

            if u == c:
                d = 0.0
                coverage_count += 1
                sum_dist += d

            else:
                d = adj_dist.get(u, {}).get(c, None)

                if d is None:
                    missing_count += 1
                else:
                    coverage_count += 1
                    sum_dist += float(d)

        coverage_ratio = coverage_count / total if total else 0.0
        mean_dist_diag = (sum_dist / coverage_count) if coverage_count > 0 else None

        cand_rows.append({
            "interest_comm_id": int(ic),
            "candidate_id": int(c),
            "sum_dist": float(sum_dist),
            "coverage_count": int(coverage_count),
            "coverage_ratio": float(coverage_ratio),
            "missing_count": int(missing_count),
            "mean_dist_diag": mean_dist_diag,
            "nominated_by_count": int(nominated_by_count_map.get(c, 0)),
            "candidate_in_IC": bool(c in Iset)
        })

    df_cand = pd.DataFrame(cand_rows)

    if WRITE_CANDIDATE_FILES:
        df_cand.to_csv(
            os.path.join(cand_dir, f"lastfm_candidate_scores_comm_{int(ic)}.csv"),
            index=False,
            encoding="utf-8-sig"
        )

        df_cand.to_excel(
            os.path.join(cand_dir, f"lastfm_candidate_scores_comm_{int(ic)}.xlsx"),
            index=False
        )

    # -----------------------------------------------------------------
    # Step D: Winner Selection
    # -----------------------------------------------------------------
    df_rank = (
        df_cand
        .sort_values(
            by=["sum_dist", "coverage_count", "candidate_id"],
            ascending=[True, False, True]
        )
        .reset_index(drop=True)
    )

    leader_row = df_rank.iloc[0].copy()

    leader = int(leader_row["candidate_id"])
    leader_sum_dist = float(leader_row["sum_dist"])
    leader_coverage_count = int(leader_row["coverage_count"])
    leader_coverage_ratio = float(leader_row["coverage_ratio"])
    leader_nominated_by_count = int(leader_row["nominated_by_count"])
    leader_in_IC = bool(leader_row["candidate_in_IC"])

    # Decision basis
    min_sum = df_rank["sum_dist"].min()
    df_sum_tied = df_rank[np.abs(df_rank["sum_dist"] - min_sum) <= EPS].copy()

    if len(df_sum_tied) == 1:
        decision_basis = "sum_distance"
        tie_group_size = 1
    else:
        max_cov = df_sum_tied["coverage_count"].max()
        df_cov_tied = df_sum_tied[df_sum_tied["coverage_count"] == max_cov].copy()

        if len(df_cov_tied) == 1:
            decision_basis = "tie_coverage"
            tie_group_size = len(df_sum_tied)
        else:
            decision_basis = "tie_id"
            tie_group_size = len(df_cov_tied)

    # Runner-up diagnostics
    if len(df_rank) >= 2:
        runner_row = df_rank.iloc[1].copy()

        runner_up = int(runner_row["candidate_id"])
        runner_up_sum_dist = float(runner_row["sum_dist"])
        runner_up_coverage_count = int(runner_row["coverage_count"])
        sum_margin = float(runner_up_sum_dist - leader_sum_dist)
        coverage_margin = int(leader_coverage_count - runner_up_coverage_count)

    else:
        runner_up = None
        runner_up_sum_dist = None
        runner_up_coverage_count = None
        sum_margin = None
        coverage_margin = None

    nominating_voters = total - abstain_count
    nomination_coverage = (nominating_voters / total) if total else 0.0

    summary_rows.append({
        "interest_comm_id": int(ic),
        "interest_size": int(total),
        "nominating_voters": int(nominating_voters),
        "abstain_count": int(abstain_count),
        "nomination_coverage": float(nomination_coverage),
        "unique_nominees": int(len(candidates)),
        "leader": int(leader),
        "leader_sum_dist": float(leader_sum_dist),
        "leader_coverage_count": int(leader_coverage_count),
        "leader_coverage_ratio": float(leader_coverage_ratio),
        "leader_nominated_by_count": int(leader_nominated_by_count),
        "leader_in_IC": bool(leader_in_IC),
        "decision_basis": decision_basis,
        "tie_group_size": int(tie_group_size),
        "runner_up": runner_up,
        "runner_up_sum_dist": runner_up_sum_dist,
        "runner_up_coverage_count": runner_up_coverage_count,
        "sum_margin": sum_margin,
        "coverage_margin": coverage_margin
    })

    # -----------------------------------------------------------------
    # Optional: within-community distance matrix
    # -----------------------------------------------------------------
    if WRITE_WITHIN_I_MATRICES:
        mat = pd.DataFrame(
            np.nan,
            index=I,
            columns=I,
            dtype=float
        )

        np.fill_diagonal(mat.values, 0.0)

        for u in I:
            for v in I:
                if u == v:
                    mat.at[u, v] = 0.0
                else:
                    d = adj_dist.get(u, {}).get(v, None)

                    if d is not None:
                        mat.at[u, v] = float(d)

        mat.to_excel(
            os.path.join(
                mat_dir,
                f"lastfm_friend_distance_within_IC_comm_{int(ic)}.xlsx"
            ),
            index=True
        )


# =====================================================================================
# Cell 5 — Final Summary
# =====================================================================================

df_summary = (
    pd.DataFrame(summary_rows)
    .sort_values("interest_comm_id")
    .reset_index(drop=True)
)

summary_csv = os.path.join(
    final_dir,
    "lastfm_scenario1_global_friend_sumdist_no_fallback_summary.csv"
)

summary_xlsx = os.path.join(
    final_dir,
    "lastfm_scenario1_global_friend_sumdist_no_fallback_summary.xlsx"
)

df_summary.to_csv(
    summary_csv,
    index=False,
    encoding="utf-8-sig"
)

df_summary.to_excel(
    summary_xlsx,
    index=False
)

print("\n[DONE] Last.fm Scenario-1 Trusted Node Election finished.")
print("[PATH] project_dir:", project_dir)
print("[PATH] vote_dir:", vote_dir)
print("[PATH] nom_dir:", nom_dir)
print("[PATH] cand_dir:", cand_dir)
print("[PATH] final_dir:", final_dir)
print("[PATH] final_summary_csv:", summary_csv)
print("[PATH] final_summary_xlsx:", summary_xlsx)

if WRITE_DISTANCE_EDGELIST:
    print("[PATH] distance_edgelist:", dist_path)

if WRITE_WITHIN_I_MATRICES:
    print("[PATH] matrices_dir:", mat_dir)

print("\n[FINAL SUMMARY]")
print(df_summary.to_string(index=False))

[INFO] RUN_ID: RUN_20260601_161913
[INFO] project_dir: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\2_التصويت\RUN_20260601_161913
[INFO] outputs_dir: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\2_التصويت\RUN_20260601_161913\outputs
[INFO] friends_sim_csv: D:\PHD\بابل\2024\Research Reco\الجزء الثاني\lastfmReco\lastfm_friend_sim_mtrx\lastfm_friends_jaccard_sim_nonzero.csv
[INFO] number of community files: 19
[OK] Input paths verified. Output folders created.
[OK] Interest communities: 19
[OK] Total unique interest users: 773

[INTEREST COMMUNITY SIZES]
 interest_comm_id  interest_comm_size
                5                  16
               28                  38
               30                  66
               38                  57
               40                  58
               44                  26
               60                 136
               99                  81
              104                  25
              107                  44
    